In [23]:
import numpy as np
import pandas as pd
embeddings = np.load('lcb-embeddings.npy')
# target = pd.read_parquet('llama323bi-code-maxk-loo-k4-lcb.parquet')
target = pd.read_parquet('llama323bi-code-maxk-dm-traink4-testk4-lr1e3-ns10-lcb.parquet')


In [24]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import norm
from scipy.integrate import quad
from typing import Sequence, Union


def _normalize_hidden(hidden: Union[int, Sequence[int]]) -> list:
    if isinstance(hidden, int):
        return [hidden]
    return list(hidden)


def _mlp_layers(
    in_features: int,
    hidden: Union[int, Sequence[int]],
    out_features: int,
) -> nn.Sequential:
    """Linear-ReLU blocks with widths ``hidden`` (single int or list), ending with Linear(out_features)."""
    hids = _normalize_hidden(hidden)
    layers: list = []
    d_in = in_features
    for h in hids:
        layers.extend([nn.Linear(d_in, h), nn.ReLU()])
        d_in = h
    layers.append(nn.Linear(d_in, out_features))
    return nn.Sequential(*layers)


# =====================================================================
# 2. NEURAL ARCHITECTURE & ESTIMATORS
# =====================================================================
class SoftmaxNet(nn.Module):
    def __init__(
        self,
        num_bins: int = 20,
        hidden: Union[int, Sequence[int]] = 64,
        in_features: int = 1,
    ):
        super().__init__()
        self.num_bins = num_bins
        self.mlp = nn.Sequential(
            _mlp_layers(in_features, hidden, num_bins),
            nn.Softmax(dim=-1),  # Directly outputs a normalized PDF
        )

    def forward(self, x: torch.Tensor):
        return self.mlp(x)


def cross_entropy_loss(predicted_pdf: torch.Tensor, target_counts: torch.Tensor):
    """
    predicted_pdf: (n_X, num_bins) from the Softmax network.
    target_counts: (n_X, num_bins) the grouped empirical counts (e.g., C_total).
    """
    eps = 1e-8
    N = target_counts.sum(dim=-1, keepdim=True)
    
    # Standard Cross-Entropy over multinomial counts
    loss = -torch.sum(target_counts * torch.log(predicted_pdf + eps), dim=-1)
    
    # Divide by N to match the scale of the flattened N=1 loss
    return (loss / N.squeeze(-1)).mean()


from sklearn.isotonic import IsotonicRegression

class SigmoidNet(nn.Module):
    def __init__(
        self,
        num_bins: int = 20,
        hidden: Union[int, Sequence[int]] = 64,
        in_features: int = 1,
    ):
        super().__init__()
        self.num_bins = num_bins
        # Outputs B-1 independent raw thresholds, NOT a normalized distribution
        self.mlp = _mlp_layers(in_features, hidden, num_bins - 1)

    def forward(self, x: torch.Tensor):
        return torch.sigmoid(self.mlp(x))


def ordinal_bce_loss(predicted_cdf: torch.Tensor, target_cdf: torch.Tensor):
    """
    predicted_cdf: (..., num_bins - 1) raw sigmoids
    target_cdf: (..., num_bins - 1) empirical CDF thresholds
    """
    # reduction='none' gives us the (n_X, num_bins - 1) matrix of errors.
    # We sum over the bins (dim=-1), then average over the batch.
    bce = F.binary_cross_entropy(predicted_cdf, target_cdf, reduction='none')
    return bce.sum(dim=-1).mean()


def apply_pava_batched(raw_cdf: torch.Tensor):
    """
    Applies exact O(N) Isotonic Regression via sklearn to enforce monotonicity.
    Must be used inside torch.no_grad().
    """
    device = raw_cdf.device
    shape = raw_cdf.shape
    
    # Flatten batch dimensions (e.g. n_X)
    raw_np = raw_cdf.detach().cpu().numpy().reshape(-1, shape[-1])
    iso_np = np.zeros_like(raw_np)
    
    # y_min/y_max mathematically bounds the outer edges of the CDF to [0, 1]
    iso_reg = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds='clip')
    x = np.arange(shape[-1])

    # Run PAVA independently on every rollout in the batch
    for i in range(raw_np.shape[0]):
        iso_np[i] = iso_reg.fit_transform(x, raw_np[i])

    return torch.tensor(iso_np, dtype=raw_cdf.dtype, device=device).view(*shape)


def compute_advantage_max_at_k(pdf, Y_t, k, NUM_BINS):
    predicted_cdf = torch.cumsum(pdf, dim=-1)[..., :-1]
    thresholds = torch.linspace(0, 1, NUM_BINS + 1, device=Y_t.device, dtype=pdf.dtype)[1:-1]
    indicator = (Y_t.unsqueeze(-1) > thresholds).float()
    baseline = 1.0 - predicted_cdf.unsqueeze(1)
    error_signal = indicator - baseline
    weight = (predicted_cdf ** (k - 1)).unsqueeze(1)
    delta_b = 1.0 / NUM_BINS
    per_action_adv = torch.sum(weight * error_signal * delta_b, dim=-1)
    return per_action_adv


In [25]:
k = 4
NUM_BINS = 20
net_lr = 0.01
wd = 0
net_hidden = [64]
inner_steps = 100

# Hold out a fraction of *problems* (rows); targets are aggregates per row.
VAL_FRAC = 0.2
VAL_SEED = 0
log_every = 10
early_stop_patience = 15  # consecutive steps without val improvement
# early_stop_patience = inner_steps # disable early stopping
early_stop_min_delta = 0.0  # val must improve by this much to reset patience

X_t = torch.tensor(embeddings, dtype=torch.float64)
targets = np.array([x for x in target['scores']])
Y_t = torch.tensor(targets)
X_DIM = X_t.shape[1]
n_Z = Y_t.shape[1]
# Empirical Bin Counts
bin_indices = torch.clamp((Y_t * NUM_BINS).long(), 0, NUM_BINS - 1)
Y_one_hot = F.one_hot(bin_indices, NUM_BINS).double() # (n_X, n_Z, bins)
empirical_pdf = Y_one_hot.mean(dim=1)

n_problems = X_t.shape[0]
rng = np.random.default_rng(VAL_SEED)
perm = rng.permutation(n_problems)
val_size = max(1, int(round(n_problems * VAL_FRAC)))
val_idx = torch.tensor(perm[:val_size])
train_idx = torch.tensor(perm[val_size:])

device = 'cpu'
X_train, pdf_train = X_t[train_idx].to(device), empirical_pdf[train_idx].to(device)
X_val, pdf_val = X_t[val_idx].to(device), empirical_pdf[val_idx].to(device)
Y_train, Y_val = Y_t[train_idx].to(device), Y_t[val_idx].to(device)
thresholds = torch.linspace(0, 1, NUM_BINS + 1, dtype=torch.float64, device=device)[1:-1]
cdf_train = (Y_train.unsqueeze(-1) <= thresholds).double().mean(dim=1)
cdf_val = (Y_val.unsqueeze(-1) <= thresholds).double().mean(dim=1)

In [26]:

# B. Smoothed CE targets (one smoothing pass): per-problem mixture of Gaussian bumps on bins,
#    then same batch size / ``cross_entropy_loss`` as plain CE (no flattened batch).
def smoothed_target_pdf(Y: torch.Tensor, num_bins: int, sigma: float) -> torch.Tensor:
    """Y: (n_problems, n_Z); returns (n_problems, num_bins) row-normalized."""
    bin_centers = torch.linspace(
        0.5 / num_bins, 1.0 - 0.5 / num_bins, num_bins, device=Y.device, dtype=Y.dtype
    )
    dist_sq = (Y.unsqueeze(-1) - bin_centers) ** 2
    w = torch.exp(-dist_sq / (2 * sigma**2))
    w = w / (w.sum(dim=-1, keepdim=True) + 1e-12)
    return w.mean(dim=1)


SCE_SIGMA = 0.05
smooth_pdf_train = smoothed_target_pdf(Y_train, NUM_BINS, SCE_SIGMA)
smooth_pdf_val = smoothed_target_pdf(Y_val, NUM_BINS, SCE_SIGMA)

# Ensure all networks are double precision
ce_net = SoftmaxNet(num_bins=NUM_BINS, hidden=net_hidden, in_features=X_DIM).double().to(device)
ce_opt = torch.optim.Adam(ce_net.parameters(), lr=net_lr, weight_decay=wd)

pava_net = SigmoidNet(num_bins=NUM_BINS, hidden=net_hidden, in_features=X_DIM).double().to(device)
pava_opt = torch.optim.Adam(pava_net.parameters(), lr=net_lr, weight_decay=wd)

sce_net = SoftmaxNet(num_bins=NUM_BINS, hidden=net_hidden, in_features=X_DIM).double().to(device)
sce_opt = torch.optim.Adam(sce_net.parameters(), lr=net_lr, weight_decay=wd)

# Extractors
def get_pdf_ce(net_out): return net_out # Already Softmax


def get_pdf_ordinal(net_out):
    cdf = apply_pava_batched(net_out)
    # Derive PDF from isotonic CDF
    zeros = torch.zeros((cdf.shape[0], 1), dtype=cdf.dtype, device=cdf.device)
    ones = torch.ones((cdf.shape[0], 1), dtype=cdf.dtype, device=cdf.device)
    full_cdf = torch.cat([zeros, cdf, ones], dim=1)
    return full_cdf[:, 1:] - full_cdf[:, :-1]

# =====================================================================
# 3. UNIVERSAL TRAINING LOOP
# =====================================================================
def train(net, opt, loss_fn, get_pdf_fn, X_tr, Y_tr, X_v, Y_v, name="Model"):
    best_val_loss = float("inf")
    patience_ctr = 0
    best_state = None

    print(f"\n--- Training {name} ---")
    for step in range(inner_steps):
        net.train()
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(net(X_tr), Y_tr)
        loss.backward()
        opt.step()

        net.eval()
        with torch.no_grad():
            train_loss = loss_fn(net(X_tr), Y_tr).item()
            val_loss = loss_fn(net(X_v), Y_v).item()

        improved = val_loss < best_val_loss - early_stop_min_delta
        if improved:
            best_val_loss = val_loss
            patience_ctr = 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            patience_ctr += 1

        if patience_ctr >= early_stop_patience:
            print(f"Early stopping at step {step}: best_val_loss={best_val_loss:.4f}")
            break

    if best_state is not None:
        net.load_state_dict(best_state)

    net.eval()
    with torch.no_grad():
        predicted_pdf_val = get_pdf_fn(net(X_v))
    return predicted_pdf_val


In [27]:
from scipy.special import gammaln
import torch
import torch.nn.functional as F

# =====================================================================
# 4. EVALUATE PREDICTION METRICS
# =====================================================================
empirical_pdf_val = pdf_val  # It's already a PDF!

# We can safely use the requested k, provided it's <= our total samples
k_adv = min(k, n_Z)

def combinatorial_true_advantage_numpy(Y: np.ndarray, k_adv: int) -> np.ndarray:
    """
    Computes the exact expected advantage of each sample in Y for a max@k_adv objective.
    It marginalizes over all subsets of size k_adv-1 drawn from the other N-1 samples.
    """
    n, N = Y.shape
    if k_adv < 2:
        # Standard REINFORCE baseline (k=1)
        return Y - np.mean(Y, axis=-1, keepdims=True)

    adv = np.zeros_like(Y)
    
    # Precompute the combinatorial weights for the max of (k-1) items drawn from (N-1) items.
    # To be the max, an item at sorted index j must have the other (k-2) items chosen from the j items below it.
    j_indices = np.arange(N - 1)
    valid = j_indices >= (k_adv - 2)
    w = np.zeros(N - 1)
    
    n_choose = j_indices[valid]
    r_choose = k_adv - 2
    
    # log(nCr) = gammaln(n+1) - gammaln(r+1) - gammaln(n-r+1)
    log_num = gammaln(n_choose + 1) - gammaln(r_choose + 1) - gammaln(n_choose - r_choose + 1)
    # Denominator is N-1 choose k-1
    log_den = gammaln(N) - gammaln(k_adv) - gammaln(N - k_adv + 1)
    
    w[valid] = np.exp(log_num - log_den)
    
    # Calculate expected marginal contribution for every sample
    for i in range(N):
        y_i = Y[:, i]
        Y_minus_i = np.delete(Y, i, axis=1)
        Y_minus_i_sorted = np.sort(Y_minus_i, axis=1)
        
        # Contribution is max(y_i - M_S, 0)
        diff = np.maximum(y_i[:, None] - Y_minus_i_sorted, 0.0)
        adv[:, i] = np.sum(diff * w, axis=1)
        
    # Mean-center the advantage to create a valid Policy Gradient score (E[Adv] = 0)
    adv = adv - np.mean(adv, axis=-1, keepdims=True)
    return adv

# Calculate the True Oracle Advantage over ALL available validation samples
if k_adv >= 2:
    true_adv_np = combinatorial_true_advantage_numpy(Y_val.detach().cpu().numpy(), k_adv)
else:
    true_adv_np = None
    print("Skipping advantage vs LOO metrics (need min(k, n_Z) >= 2).")

models = [
    (ce_net, ce_opt, cross_entropy_loss, get_pdf_ce, pdf_train, pdf_val, 'CE'),
    (pava_net, pava_opt, ordinal_bce_loss, get_pdf_ordinal, cdf_train, cdf_val, 'Ordinal BCE'),
    (sce_net, sce_opt, cross_entropy_loss, get_pdf_ce, smooth_pdf_train, smooth_pdf_val, 'Smoothed CE'),
]

print("\n=== FINAL VALIDATION METRICS ===")
for (net, opt, loss_fn, get_pdf_func, target_tr, target_v, name) in models:
    pred_pdf_val = train(net, opt, loss_fn, get_pdf_func, X_train, target_tr, X_val, target_v, name=name)

    # 1. Distribution Metric
    pdf_err = F.mse_loss(pred_pdf_val, empirical_pdf_val).item()
    print(f"[{name}] Validation PDF MSE: {pdf_err:.6f}")

    # 2. Advantage Metric
    if true_adv_np is not None:
        # We evaluate the predicted advantage over ALL n_Z samples simultaneously
        pred_adv_t = compute_advantage_max_at_k(pred_pdf_val, Y_val, k_adv, NUM_BINS)
        pred_adv_np = pred_adv_t.detach().cpu().numpy()
        
        # Mean-center the predicted advantage to align with combinatorial oracle (global mean of diff ~0).
        pred_adv_np = pred_adv_np - np.mean(pred_adv_np, axis=-1, keepdims=True)
        
        d_adv = pred_adv_np - true_adv_np
        print(f"[{name}] Pred vs combinatorial oracle adv -- MAE: {np.mean(np.abs(d_adv)):.6f}, "
              f"RMSE: {np.sqrt(np.mean(d_adv**2)):.6f}\n")



=== FINAL VALIDATION METRICS ===

--- Training CE ---
Early stopping at step 52: best_val_loss=0.4469
[CE] Validation PDF MSE: 0.005906
[CE] Pred vs combinatorial oracle adv -- MAE: 0.020957, RMSE: 0.061265


--- Training Ordinal BCE ---
Early stopping at step 49: best_val_loss=5.2285
[Ordinal BCE] Validation PDF MSE: 0.005257
[Ordinal BCE] Pred vs combinatorial oracle adv -- MAE: 0.022851, RMSE: 0.067757


--- Training Smoothed CE ---
Early stopping at step 56: best_val_loss=1.1623
[Smoothed CE] Validation PDF MSE: 0.011779
[Smoothed CE] Pred vs combinatorial oracle adv -- MAE: 0.021680, RMSE: 0.062656

